In [1]:
import os
import sys
import numpy as np
import subprocess
from shutil import which

In [2]:
# 0..21 から 21(=index 20) を除くチーム一覧
# prep: 1-22 (21 is missing)
# contest: 1-20,22-24
def get_teams():
    arr = np.arange(1, 25)
    return np.delete(arr, 20)  # 21 をスキップ（コンテスト仕様に合わせる）

In [3]:
# testing for team 22 only
def get_teams():
    arr = np.arange(22, 23)
    return arr  

In [4]:
def loop_for_all_teams(
    command_template,
    *,
    dry_run=False,
    strict=True,
    continue_on_error=False,
    cwd=None
):
    """
    command_template: ['python', 'attack/attack_Ci.py', '...{id:02d}...', ...] のようなリスト
    dry_run: True -> 実行せず展開コマンドのみ表示
    strict: True -> {id:02d} が一つも無ければ例外
    continue_on_error: True -> 失敗しても次の team へ。False -> そこで中断
    cwd: サブプロセスの作業ディレクトリ（attack/ ディレクトリの相対パス解決に使える）
    """
    id_indices = [i for i, arg in enumerate(command_template)
                  if isinstance(arg, str) and "{id:02d}" in arg]
    if strict and not id_indices:
        raise ValueError(f"No {{id:02d}} placeholder found in: {command_template}")

    # 'python' を実行ファイルに置換（環境ズレ回避）
    cmd0 = command_template[:]
    if cmd0 and cmd0[0] in ("python", "python3"):
        cmd0[0] = sys.executable

    # 事前: 実行ファイルの存在チェック（python 以外の最初の実体コマンドにも対応）
    exe = cmd0[0]
    if os.path.sep not in exe and which(exe) is None:
        raise RuntimeError(f"Executable not found on PATH: {exe}")

    for team in get_teams():
        cmd = cmd0[:]
        for ind in id_indices:
            cmd[ind] = cmd[ind].format(id=team)

        # 簡易プリフライト: 既知の入力系ファイルっぽい引数を存在確認
        # （.csv, .json かつ -o/--out* ではない位置を対象にする）
        def is_out_flag(i):
            return isinstance(cmd[i-1], str) and (
                cmd[i-1] in ("-o", "--out", "--out-map", "--out-pred", "--out-conf")
                or cmd[i-1].startswith("--out")
            )

        missing_inputs = []
        for i, a in enumerate(cmd):
            if isinstance(a, str) and (a.endswith(".csv") or a.endswith(".json")):
                if not is_out_flag(i):  # 出力ではなく入力と推定
                    apath = a if cwd is None else os.path.join(cwd, a)
                    if not os.path.exists(apath):
                        missing_inputs.append(a)

        print(">>", " ".join(cmd))
        if missing_inputs:
            msg = f"[team {team}] Missing input files: {missing_inputs}"
            if continue_on_error:
                print("!!", msg)
                continue
            else:
                raise FileNotFoundError(msg)

        if dry_run:
            continue

        try:
            # 標準出力・標準エラーを取得して、失敗時に見せる
            completed = subprocess.run(
                cmd, check=True, cwd=cwd,
                capture_output=True, text=True
            )
            if completed.stdout:
                print(completed.stdout.strip())
        except subprocess.CalledProcessError as e:
            print(f"\n[ERROR] team {team} command failed with code {e.returncode}")
            if e.stdout:
                print("--- stdout ---")
                print(e.stdout.strip())
            if e.stderr:
                print("--- stderr ---")
                print(e.stderr.strip())
            if not continue_on_error:
                raise

In [5]:
def loop_for_all_teams_and_variants(
    command_template,
    variants,
    *,
    dry_run=False,
    strict=True,
    continue_on_error=False,
    cwd=None
):
    """
    command_template: ['python', ..., 'in/BB{id:02d}_{variant}.csv', ...] のようなリスト
    variants: ['1', '2', '3'] など
    """
    id_indices = [i for i, arg in enumerate(command_template)
                  if isinstance(arg, str) and "{id:02d}" in arg]
    variant_indices = [i for i, arg in enumerate(command_template)
                       if isinstance(arg, str) and "{variant}" in arg]
    if strict and not (id_indices or variant_indices):
        raise ValueError("No {id:02d} or {variant} placeholder found in: {}".format(command_template))

    cmd0 = command_template[:]
    if cmd0 and cmd0[0] in ("python", "python3"):
        cmd0[0] = sys.executable

    exe = cmd0[0]
    if os.path.sep not in exe and which(exe) is None:
        raise RuntimeError(f"Executable not found on PATH: {exe}")

    for team in get_teams():
        for variant in variants:
            cmd = []
            for arg in cmd0:
                if isinstance(arg, str):
                    cmd.append(arg.format(id=team, variant=variant))
                else:
                    cmd.append(arg)

            def is_out_flag(i):
                return isinstance(cmd[i-1], str) and (
                    cmd[i-1] in ("-o", "--out", "--out-map", "--out-pred", "--out-conf")
                    or cmd[i-1].startswith("--out")
                )

            missing_inputs = []
            for i, a in enumerate(cmd):
                if isinstance(a, str) and (a.endswith(".csv") or a.endswith(".json")):
                    if not is_out_flag(i):
                        apath = a if cwd is None else os.path.join(cwd, a)
                        if not os.path.exists(apath):
                            missing_inputs.append(a)

            print(">>", " ".join(cmd))
            if missing_inputs:
                msg = f"[team {team} variant {variant}] Missing input files: {missing_inputs}"
                if continue_on_error:
                    print("!!", msg)
                    continue
                else:
                    raise FileNotFoundError(msg)

            if dry_run:
                continue

            try:
                completed = subprocess.run(
                    cmd, check=True, cwd=cwd,
                    capture_output=True, text=True
                )
                if completed.stdout:
                    print(completed.stdout.strip())
            except subprocess.CalledProcessError as e:
                print(f"\n[ERROR] team {team} variant {variant} command failed with code {e.returncode}")
                if e.stdout:
                    print("--- stdout ---")
                    print(e.stdout.strip())
                if e.stderr:
                    print("--- stderr ---")
                    print(e.stderr.strip())
                if not continue_on_error:
                    raise

In [6]:
# 必要なら cwd='プロジェクトのルート' を指定（例: cwd=r'c:\work\pwscup2025'）
cwd = r"/home/kikuchih/pwscup2025-scripts"  # <- 適宜書き換え
os.chdir(cwd)

In [7]:
## Create input folder and Place "Original Datasets(B**/BB**)".
### create "in/"
### place B22_1, etc. in "in/"
if not os.path.exists("in"):
    os.makedirs("in")

In [8]:
prep_original = "B"
contest_original = "BB"
prep_anon = "C"
contest_anon = "CC"
prep_model = "D"
contest_model = "DD"

In [9]:
# choose mode(prep or contest)
mode = "contest"

if mode == "prep":
    mode_original = prep_original
    mode_anon = prep_anon
    mode_model = prep_model
else:
    mode_original = contest_original
    mode_anon = contest_anon
    mode_model = contest_model

In [10]:
## Creation of Ci with original sample code
variants = ['1', '2', '3']
Ci_anonymization_original = [
    "python", "anonymization/ano_fixed.py",
    f"in/{mode_original}" + "{id:02d}_{variant}.csv",
    "-o", 
    f"in/{mode_anon}" + "{id:02d}_{variant}.csv",
    "--seed", "42"
]
loop_for_all_teams_and_variants(Ci_anonymization_original, variants)

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/ano_fixed.py in/BB22_1.csv -o in/CC22_1.csv --seed 42
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/ano_fixed.py in/BB22_2.csv -o in/CC22_2.csv --seed 42
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/ano_fixed.py in/BB22_3.csv -o in/CC22_3.csv --seed 42


In [11]:
## Creation of Di with original samples(Bi only)
variants = ['1', '2', '3']
Di_anonymization_original_Bi = ["python", "analysis/xgbt_train_fixed.py", f"in/{mode_original}"+"{id:02d}_{variant}.csv", "-o", f"in/{mode_model}"+"{id:02d}_Bi_{variant}.json"]
loop_for_all_teams_and_variants(Di_anonymization_original_Bi, variants)
print("sample Di-anonymization(Bi only) completed")


>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python analysis/xgbt_train_fixed.py in/BB22_1.csv -o in/DD22_Bi_1.json
Validation Accuracy (threshold=0.5): 0.876000
Saved model JSON to: in/DD22_Bi_1.json
#features: 21
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python analysis/xgbt_train_fixed.py in/BB22_2.csv -o in/DD22_Bi_2.json
Validation Accuracy (threshold=0.5): 0.883000
Saved model JSON to: in/DD22_Bi_2.json
#features: 21
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python analysis/xgbt_train_fixed.py in/BB22_3.csv -o in/DD22_Bi_3.json
Validation Accuracy (threshold=0.5): 0.885000
Saved model JSON to: in/DD22_Bi_3.json
#features: 21
sample Di-anonymization(Bi only) completed


In [12]:
## Creation of Di with original samples(Ci only)
variants = ['1', '2', '3']
Di_anonymization_original_Ci = ["python", "analysis/xgbt_train_fixed.py", f"in/{mode_anon}"+"{id:02d}_{variant}.csv", "-o", f"in/{mode_model}"+"{id:02d}_Ci_{variant}.json"]
loop_for_all_teams_and_variants(Di_anonymization_original_Ci, variants)
print("sample Di-anonymization(Ci only) completed")

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python analysis/xgbt_train_fixed.py in/CC22_1.csv -o in/DD22_Ci_1.json
Validation Accuracy (threshold=0.5): 0.746000
Saved model JSON to: in/DD22_Ci_1.json
#features: 21
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python analysis/xgbt_train_fixed.py in/CC22_2.csv -o in/DD22_Ci_2.json
Validation Accuracy (threshold=0.5): 0.747000
Saved model JSON to: in/DD22_Ci_2.json
#features: 21
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python analysis/xgbt_train_fixed.py in/CC22_3.csv -o in/DD22_Ci_3.json
Validation Accuracy (threshold=0.5): 0.729000
Saved model JSON to: in/DD22_Ci_3.json
#features: 21
sample Di-anonymization(Ci only) completed


In [13]:
## Creation of Di with original samples(Bi and Ci)
variants = ['1', '2', '3']
Di_anonymization_original = ["python", "anonymization/gen_Di_fixed.py", f"in/{mode_original}"+"{id:02d}_{variant}.csv", f"in/{mode_anon}"+"{id:02d}_{variant}.csv", "-o", f"in/{mode_model}"+"{id:02d}_BiCi_{variant}.json"]
loop_for_all_teams_and_variants(Di_anonymization_original, variants)
print("sample Ci-anonymization completed")

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/gen_Di_fixed.py in/BB22_1.csv in/CC22_1.csv -o in/DD22_BiCi_1.json
accuracy: 0.8931
a Di.json example was saved as in/DD22_BiCi_1.json
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/gen_Di_fixed.py in/BB22_2.csv in/CC22_2.csv -o in/DD22_BiCi_2.json
accuracy: 0.8885
a Di.json example was saved as in/DD22_BiCi_2.json
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/gen_Di_fixed.py in/BB22_3.csv in/CC22_3.csv -o in/DD22_BiCi_3.json
accuracy: 0.8786
a Di.json example was saved as in/DD22_BiCi_3.json
sample Ci-anonymization completed


In [24]:
## mondorian anonymization
variants = ['1', '2', '3']
mondorian_anonymization_original = [
    "python", "third_party/k-anonymity/anonymize-pws.py",
    "--method", "classic_mondrian",
    "--k", "10",
    "--input", f"in/{mode_original}" + "{id:02d}_{variant}.csv",
    "-o", f"in/{mode_anon}" + "{id:02d}_{variant}_mondorian_k10.csv"
]
loop_for_all_teams_and_variants(mondorian_anonymization_original, variants)

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python third_party/k-anonymity/anonymize-pws.py --method classic_mondrian --k 10 --input in/BB22_1.csv -o in/CC22_1_mondorian_k10.csv



[ERROR] team 22 variant 1 command failed with code 1
--- stderr ---
Traceback (most recent call last):
  File "/home/kikuchih/pwscup2025-scripts/third_party/k-anonymity/anonymize-pws.py", line 172, in <module>
    main(args)
  File "/home/kikuchih/pwscup2025-scripts/third_party/k-anonymity/anonymize-pws.py", line 167, in main
    anonymizer.anonymize()
  File "/home/kikuchih/pwscup2025-scripts/third_party/k-anonymity/anonymize-pws.py", line 79, in anonymize
    QI_NAMES = list(np.array(ATT_NAMES)[QI_INDEX])
                    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^
IndexError: index 18 is out of bounds for axis 0 with size 18


CalledProcessError: Command '['/home/kikuchih/miniconda3/envs/pwscup2025/bin/python', 'third_party/k-anonymity/anonymize-pws.py', '--method', 'classic_mondrian', '--k', '10', '--input', 'in/BB22_1.csv', '-o', 'in/CC22_1_mondorian_k10.csv']' returned non-zero exit status 1.

In [18]:
## DataSynthesizer (複数epsilon/k対応)
epsilon_list = [1.0, 2.0]  # 必要な値に変更
k_list = [2, 3]            # 必要な値に変更
variants = ['1', '2', '3']

for epsilon in epsilon_list:
    for k in k_list:
        DataSynthesizer_anonymization_original = [
            "python",
            "anonymization/ano_DataSynthesizer.py",
            f"in/{mode_original}" + "{id:02d}_{variant}.csv",
            "-o",
            f"in/{mode_anon}" + f"{{id:02d}}_{{variant}}_ds_e{epsilon}_k{k}.csv",
            "--epsilon", str(epsilon),
            "--k", str(k),
            "--mode", "correlated_attribute_mode",  # 必要に応じて変更(['correlated_attribute_mode', 'independent_attribute_mode', 'random_mode'])
            "--num_tuples", "10000",
            "--seed", "42",
            # "--desc",  f"in/{mode_anon}" + f"{{id:02d}}_{{variant}}_ds_e{epsilon}_k{k}.json",
            # "--edges", f"in/{mode_anon}" + f"{{id:02d}}_{{variant}}_ds_e{epsilon}_k{k}.pkl",
        ]
        loop_for_all_teams_and_variants(DataSynthesizer_anonymization_original, variants)

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python anonymization/ano_DataSynthesizer.py in/BB22_1.csv -o in/CC22_1_ds_e1.0_k2.csv --epsilon 1.0 --k 2 --mode correlated_attribute_mode --num_tuples 10000 --seed 42


================ Constructing Bayesian Network (BN) ================
Adding ROOT ETHNICITY
Adding attribute num_immunizations
Adding attribute mean_weight
Adding attribute AGE
Adding attribute obesity_flag
Adding attribute num_procedures
Adding attribute encounter_count
Adding attribute RACE
Adding attribute mean_bmi
Adding attribute stroke_flag
Adding attribute asthma_flag
Adding attribute GENDER
Adding attribute mean_diastolic_bp
Adding attribute mean_systolic_bp
Adding attribute num_medications
Adding attribute num_allergies
Adding attribute num_devices
Adding attribute depression_flag
========================== BN constructed ==========================
Constructed Bayesian network:
    num_immunizations has parents ['ETHNICITY'].
    mean_weight       has parents ['num_immunizations', 'ETHNICITY'].
    AGE               has parents ['mean_weight', 'num_immunizations'].
    obesity_flag      has parents ['AGE', 'mean_weight'].
    num_procedures    has parents ['obesity_flag', 'mean